# shelf-vision: fine-tune the detector on Kaggle

Short annealing run on top of a finished training run. The 11.32h run stopped
early at epoch 83 of 146, with the LR still high and without the mosaic-free
final epochs; this run does that part (`configs/detect/yolo11s-sku110k-ft.yaml`,
12 epochs, about 1 hour of training).

**Before running:**
- Settings → Accelerator **GPU T4 x2**, Internet **on**.
- Add Input → Notebooks → the training notebook whose output has
  `runs/detect/yolo11s-sku110k-*h/weights/best.pt`.
- **Save Version → Save & Run All (Commit)**.

In [ ]:
import glob
import os
import shutil

DATA_DIR = "/tmp/shelf-data"  # ~24 GB peak; /kaggle/working is limited to ~20 GB
os.environ["SHELF_DATA_DIR"] = DATA_DIR

found = sorted(glob.glob("/kaggle/input/**/yolo11s-sku110k-*h/weights/best.pt", recursive=True))
assert found, "attach the training notebook's output as an input (see above)"
BASE = found[-1]
print("starting from", BASE)

!nvidia-smi --query-gpu=name,memory.total --format=csv
free_gb = shutil.disk_usage("/tmp").free / 2**30
assert free_gb > 30, f"only {free_gb:.0f} GB free in /tmp, need ~30 GB for the dataset"

In [ ]:
!git clone -q https://github.com/rsuvbe/shelf-vision
!git -C shelf-vision log --oneline -1
!pip install -q -e shelf-vision
!shelf doctor

## Data

In [ ]:
!bash shelf-vision/scripts/fetch_sku110k.sh {DATA_DIR}
!shelf prepare shelf-vision/configs/detect/yolo11s-sku110k-ft.yaml
!rm -f {DATA_DIR}/SKU110K_fixed.tar.gz

## Fine-tune

In [ ]:
!shelf train shelf-vision/configs/detect/yolo11s-sku110k-ft.yaml --weights {BASE} 2>&1 | tr '\r' '\n' | grep -v '━' | tee /kaggle/working/train.log

## Test set

Both the starting weights and the fine-tuned ones, evaluated in the same
session so the comparison is like for like.

In [ ]:
FT = max(glob.glob("/kaggle/working/runs/detect/yolo11s-sku110k-ft*/weights/best.pt"))
for name, w in [("base", BASE), ("fine-tuned", FT)]:
    print(f"== {name}: {w}")
    !shelf evaluate {w} shelf-vision/configs/detect/yolo11s-sku110k-ft.yaml --split test 2>&1 | tr '\r' '\n' | grep -v '━' | tail -3 | tee -a /kaggle/working/train.log

Output to download: `runs/detect/yolo11s-sku110k-ft/` (weights, `results.csv`,
plots) and `train.log`.